In [2]:
import json
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoModelForCausalLM

import time

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
# ============================================================
# CONFIG
# ============================================================
path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
with open(path, "r") as f:
    dataset_days = json.load(f)

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

countries = ["Denmark"]

days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
    "price_eur_kwh"
]

LOOKBACK = 2880          # max supported context length from example
PRED_LEN = 96            # 96 steps ahead

# ============================================================
# LOAD TIMER MODEL
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = AutoModelForCausalLM.from_pretrained(
    "thuml/timer-base-84m",
    trust_remote_code=True
).to(device)

model.eval()

rmse_results = []

# ============================================================
# MAIN LOOP
# ============================================================
for country in countries:
    print("Processing country:", country)

    data_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    # households are everything except weather features
    households = [col for col in df.columns if col not in features]

    # infer sampling frequency from dataframe index
    inferred_freq = pd.infer_freq(df.index)

    if inferred_freq is None:
        # fallback: use median time difference
        diffs = df.index.to_series().diff().dropna()
        step = diffs.median()
    else:
        step = pd.tseries.frequencies.to_offset(inferred_freq)

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            s_train = df.loc[df.index < cutoff, household].dropna()

            # need enough history for Timer
            if len(s_train) < LOOKBACK:
                print(f"      Skipping {household}: not enough history ({len(s_train)} < {LOOKBACK})")
                continue

            # last LOOKBACK values only
            context = s_train.iloc[-LOOKBACK:].astype("float32").values
            seqs = torch.tensor(context, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                output = model.generate(seqs, max_new_tokens=PRED_LEN)

            y_pred = output.squeeze(0).detach().cpu().numpy()

            # build forecast timestamps starting from cutoff
            pred_index = pd.date_range(start=cutoff, periods=PRED_LEN, freq=step)

            # initialize predictions df once
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=pred_index)

            predictions_df_all_households[household] = y_pred

            # true values for same timestamps
            y_true = df.reindex(pred_index)[household].values

            # remove any NaNs before RMSE
            mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
            if mask.sum() == 0:
                print(f"      Warning: no valid truth values for {household} at {day}")
                continue

            rmse = root_mean_squared_error(y_true[mask], y_pred[mask])
            rmse_households.append(rmse)

        if len(rmse_households) == 0:
            avg_rmse_households = np.nan
        else:
            avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        # save predictions for this (country, day)
        output_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        if predictions_df_all_households is not None:
            predictions_df_all_households.to_csv(output_path, index=True)
            print("      Saved:", output_path)
        else:
            print("      No predictions generated for this split.")

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

end_time = time.time()
total_seconds = end_time - start_time
print(f"Total runtime: {total_seconds:.2f} seconds")

Using device: cuda


c:\Users\CR58XM\AppData\Local\anaconda3\envs\timer\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\CR58XM\AppData\Local\anaconda3\envs\timer\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing country: Denmark
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_day1_Denmark.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_day2_Denmark.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_day3_Denmark.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_day4_Denmark.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimerUnivar_pred_day5_Denmark.csv

Per-day RMSE:
   country   day      rmse
0  Denmark  day1  1.248531
1  Denmark  day2  0.639335
2  Denmark  day3  0.615546
3  Denmark  day4  0.460315
4  Denmark  day5  0.961349

Cross-validated RMSE per country (mean over days):
country
Denmark    0.785015


In [3]:
print(f"Time taken: {total_seconds:.4f} seconds")


file_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\time_spend.json"
# 1. Load existing JSON
with open(file_path, "r") as f:
    data = json.load(f)

# 2. Add model inside "Local"
data["Foundational"]["Timer"] = total_seconds

# 3. Save back (without disturbing structure)
with open(file_path, "w") as f:
    json.dump(data, f, indent=4)

Time taken: 5.4397 seconds
